# Lens Distortion Correction — Kaggle Training & Submission

This notebook runs on Kaggle GPU (T4/P100). Steps:
1. Install dependencies & setup paths
2. Phase 1: Train parametric-only model
3. Phase 2: Train full pipeline (parametric + residual flow)
4. Run inference on test images
5. Package & submit

## 0. Setup

In [ ]:
!pip install -q timm kornia albumentations scikit-image

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

# On Kaggle, the competition data is mounted at /kaggle/input/
KAGGLE_INPUT = Path("/kaggle/input/automatic-lens-correction")
WORKING = Path("/kaggle/working")

# Clone or upload the project code to /kaggle/working/code/
# If uploaded as a Kaggle dataset, adjust this path:
CODE_ROOT = WORKING / "code"
sys.path.insert(0, str(CODE_ROOT))

print("Input contents:", list(KAGGLE_INPUT.iterdir()) if KAGGLE_INPUT.exists() else "NOT FOUND")
print(f"GPU: {!os.system('nvidia-smi -L')}")

In [ ]:
# Override config paths for Kaggle environment
import config

config.TRAIN_DIR = KAGGLE_INPUT / "lens-correction-train-clea"
config.TEST_DIR = KAGGLE_INPUT / "test-originals"
config.OUTPUT_DIR = WORKING / "outputs"
config.CHECKPOINT_DIR = config.OUTPUT_DIR / "checkpoints"
config.SUBMISSION_DIR = config.OUTPUT_DIR / "submissions"

config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config.SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# Verify data exists
train_dirs = list(config.TRAIN_DIR.iterdir()) if config.TRAIN_DIR.exists() else []
test_files = list(config.TEST_DIR.glob("*.jpg")) if config.TEST_DIR.exists() else []
print(f"Train samples: {len(train_dirs)}")
print(f"Test images: {len(test_files)}")

## 1. Phase 1 — Parametric Only

In [ ]:
import torch
from training.train import train_phase

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Adjust batch size / epochs for Kaggle GPU memory if needed
phase1 = config.PHASE1
# Uncomment to tweak:
# phase1.batch_size = 6
# phase1.epochs = 40

model = train_phase(phase1, device)

## 2. Phase 2 — Full Pipeline (Parametric + Residual Flow)

In [ ]:
phase2 = config.PHASE2
# Uncomment to tweak:
# phase2.batch_size = 2
# phase2.resolution = 512  # lower if OOM at 768
# phase2.epochs = 20

model = train_phase(phase2, device, model=model)

## 3. Quick Validation Check

Run proxy scoring on a small subset to sanity-check before full inference.

In [ ]:
from torch.utils.data import DataLoader
from data.dataset import LensTrainDataset
from losses.combined import CombinedLoss
from training.validate import run_validation

val_ds = LensTrainDataset(resolution=phase2.resolution, split="val", augment=False)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
criterion = CombinedLoss(config.PHASE2_LOSS)

val_results = run_validation(
    model, val_loader, criterion, device,
    compute_proxy=True, max_proxy_samples=50,
)

print("\nValidation results:")
for k, v in sorted(val_results.items()):
    print(f"  {k:30s}: {v:.4f}")

## 4. Inference on Test Images

In [ ]:
from inference.predict import run_inference

best_ckpt = str(config.CHECKPOINT_DIR / "phase2_full_best.pt")
submission_dir = str(config.SUBMISSION_DIR / "final")

run_inference(
    checkpoint_path=best_ckpt,
    output_dir=submission_dir,
    resolution=phase2.resolution,
    do_refine=True,
    batch_size=4,
)

## 5. Visualize a Few Results

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

sub_path = Path(submission_dir)
sample_files = sorted(sub_path.glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, f in zip(axes.flat, sample_files):
    img = cv2.imread(str(f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f.stem[:20], fontsize=8)
    ax.axis("off")
plt.suptitle("Sample Corrected Outputs", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Package Submission Zip

In [ ]:
zip_path = config.SUBMISSION_DIR / "submission.zip"
sub_dir = Path(submission_dir)
jpg_files = sorted(sub_dir.glob("*.jpg"))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
    for f in jpg_files:
        zf.write(f, f.name)

size_mb = zip_path.stat().st_size / 1e6
print(f"Created {zip_path} ({size_mb:.1f} MB, {len(jpg_files)} images)")
print(f"\nNext steps:")
print(f"  1. Upload {zip_path.name} to bounty.autohdr.com for scoring")
print(f"  2. Download the submission.csv from the scoring service")
print(f"  3. Submit the CSV to Kaggle")

## 7. (Optional) Download Checkpoint for Later Use

In [ ]:
# Copy best checkpoint to /kaggle/working for easy download
ckpt_src = config.CHECKPOINT_DIR / "phase2_full_best.pt"
ckpt_dst = WORKING / "best_model.pt"
if ckpt_src.exists():
    shutil.copy2(ckpt_src, ckpt_dst)
    print(f"Checkpoint saved to {ckpt_dst} ({ckpt_dst.stat().st_size / 1e6:.1f} MB)")